In [ ]:
import cv2
import matplotlib.pyplot as plt

# 1. 读取图片 (根据您的路径)
image_path = r".\inference\test_image.jpg"
img = cv2.imread(image_path)

if img is None:
    print(f"❌ 错误：找不到图片 {image_path}")
else:
    # 转为 RGB (Matplotlib 显示用)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 2. ✅ 您提供的中心点坐标 (共 6 个)
    centers = [
        (798, 603),   # 1
        (1223, 900),  # 2
        (1282, 1461), # 3
        (2521, 543),  # 4
        (3275, 1032), # 5
        (3376, 1347)  # 6
    ]

    # 3. ✅【再次调大】现在每个框的宽高都至少在 200~260 像素之间。
    # 如果您觉得某个框偏左或偏右，微调这里的 (宽, 高) 即可。
    sizes = [
        (220, 250), # 1号: 宽220, 高250
        (230, 230), # 2号: 宽230, 高230
        (220, 240), # 3号: 宽220, 高240
        (200, 220), # 4号: 宽200, 高220
        (210, 240), # 5号: 宽210, 高240
        (240, 260)  # 6号: 宽240, 高260
    ]

    # 4. 颜色分配逻辑 (1,2,3是红色煤气罐，4,5,6是绿色灭火器)
    labels_and_colors = [
        ("GasCyl", "red"),    # 第1个
        ("GasCyl", "red"),    # 第2个
        ("FireExt", "green"),    # 第3个
        ("FireExt", "green"), # 第4个
        ("GasCyl", "red"), # 第5个
        ("GasCyl", "red")  # 第6个
    ]

    # 5. 核心绘制逻辑
    for i in range(6):
        x_c, y_c = centers[i]   # 中心点
        w, h = sizes[i]         # 宽和高
        label, class_color = labels_and_colors[i]

        # 转换：左上角坐标 = 中心点坐标 - (宽/2, 高/2)
        x = int(x_c - w / 2)
        y = int(y_c - h / 2)

        # 设定颜色 (RGB)
        color = (255, 0, 0) if class_color == 'red' else (0, 255, 0)

        # 画矩形框 (线宽加到 5，让边框非常明显)
        cv2.rectangle(img_rgb, (x, y), (x + w, y + h), color, thickness=5)

        # 画文字标签底色
        font = cv2.FONT_HERSHEY_SIMPLEX
        (text_w, text_h), baseline = cv2.getTextSize(label, font, 0.8, 2)
        cv2.rectangle(img_rgb, (x, y - text_h - 10), (x + text_w + 5, y), color, -1)
        # 写白色文字
        cv2.putText(img_rgb, label, (x, y - 5), font, 0.8, (255, 255, 255), 2)

    # 6. 在终端显示结果
    plt.figure(figsize=(14, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.show()


In [ ]:
import cv2
import matplotlib.pyplot as plt

# ---------------- 1. 读取图像 ----------------
image_path = r".\inference\test_image.jpg"
orig_img = cv2.imread(image_path)

if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # ---------------- 2. 构造您的数据 (模拟 detections 列表) ----------------
    # 您提供的中心点
    centers = [
        (798, 603), (1223, 900), (1282, 1461),
        (2521, 543), (3275, 1032), (3376, 1347)
    ]
    # 您提供的宽高
    sizes = [
        (220, 250), (230, 230), (220, 240),
        (200, 220), (210, 240), (240, 260)
    ]
    # 您分配的颜色 (1,2,6为红GasCyl, 3,4,5为绿FireExt)
    labels_and_colors = [
        ("GasCyl", "red"), ("GasCyl", "red"), ("FireExt", "green"),
        ("FireExt", "green"), ("GasCyl", "red"), ("GasCyl", "red")
    ]

    # 将数据转换为第二段代码需要的字典列表 (detections)
    detections = []
    for i in range(6):
        x_c, y_c = centers[i]
        w, h = sizes[i]
        label, color_name = labels_and_colors[i]

        # 中心点转左上角和右下角坐标
        x1 = int(x_c - w / 2)
        y1 = int(y_c - h / 2)
        x2 = int(x_c + w / 2)
        y2 = int(y_c + h / 2)

        detections.append({
            'bbox': (x1, y1, x2, y2),
            'class_name': label,
            'color_name': color_name
        })

    # ---------------- 3. 复制图像用于绘制 ----------------
    img_bgr = orig_img.copy()

    # 设定类别对应的颜色字典
    class_colors = {
        "GasCyl": (0, 0, 255),   # OpenCV是BGR格式，所以红色是 (0,0,255)
        "FireExt": (0, 255, 0)   # 绿色是 (0,255,0)
    }

    # ---------------- 4. 绘制检测结果 (第二段代码核心逻辑) ----------------
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别和颜色
        class_name = det.get('class_name', 'Unknown')
        color_name = det.get('color_name', 'green')

        # 获取颜色 (如果是红色就用红框，如果是绿色就用绿框)
        color = class_colors.get(class_name, (0, 255, 0))

        # 绘制边界框 (第二段代码使用线宽 2)
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)

        # 绘制标签
        label = f"{class_name}"
        # 获取文本尺寸
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)

        # 绘制标签背景
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
        # 绘制标签文字
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        print(f"   [{i+1}] {class_name}: box=({x1},{y1},{x2},{y2})")

    # ---------------- 5. 转换回 RGB 用于显示 ----------------
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # ---------------- 6. 在终端显示图像 (不保存) ----------------
    plt.figure(figsize=(14, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    print(f"\n✅ 图像已在终端显示完成！(代码沿用第二段逻辑，结果与第一段完全一致)")

In [ ]:
import cv2
import matplotlib.pyplot as plt

image_path = r".\inference\test_image.jpg"

categories = ["GasCyl", "FireExt"]

class_colors = {
    "GasCyl": (0, 0, 255),
    "FireExt": (0, 255, 0)
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]
class_ids = [0, 0, 1, 1, 0, 0]

detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]

    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)

    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00
    })

orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            color = (0, 255, 0)

        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)

        label = f"{class_name}: {confidence:.2f}"
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)

        cv2.rectangle(img_bgr, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

In [ ]:
import cv2
import matplotlib.pyplot as plt


image_path = r".\inference\test_image.jpg"
categories = ["GasCyl", "FireExt"]
class_colors = {
    "GasCyl": (0, 0, 255),   # 红色 BGR
    "FireExt": (0, 255, 0)   # 绿色 BGR
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]

class_ids = [0, 0, 1, 1, 0, 0]
detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]
    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)
    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00  # 模拟置信度
    })


# 1. 读取图像【读取图像，首先我们编写读取图片数据的代码用于读取指定路径中事先准备好的图像】
orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    # 替换为您原先环境下的返回值
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # 2. 复制图像用于绘制【为了不损坏原始图像数据，我们利用代码复制一份相同的数据用来进行图像绘制操作】
    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 绘制检测结果
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框（注意：可能是 'bbox' 或 'box'）
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别信息
        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        # 获取类别名称
        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        # 获取颜色
        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            # 使用随机颜色或默认绿色
            color = (0, 255, 0)

        # 绘制边界框
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 2)

        # 绘制标签
        label = f"{class_name}: {confidence:.2f}"
        # 获取文本尺寸
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)

        # 绘制标签背景
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
        # 绘制标签文字
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    # 4. 转换回 RGB 用于显示【为了能让检测完毕的图像能够进行显示，将文件转回为RGB色彩空间，我们进行用于转换的代码编写】
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 显示图像【文件已转换完毕，下面进行显示图像的代码编写，这5行代码实现了 画布创建 → 图像渲染 → 界面美化 → 信息标注 → 最终呈现 的完整显示流程，是可视化模块中连接数据处理与人眼观察的关键环节】
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    # 6. 保存结果【显示图像的代码已编写完成，下面将结果保存，首先定义输出路径，其次将路径输出的图片保存到指定文件，最后返回图像路径，这4行代码实现了 保存结果到文件 和 返回数据给调用方 两个功能】
    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

# 返回变量 (如果是在函数里)
# return img_bgr

In [ ]:
import cv2
import matplotlib.pyplot as plt


image_path = r".\inference\test_image.jpg"
categories = ["GasCyl", "FireExt"]
class_colors = {
    "GasCyl": (0, 0, 255),   # 红色 BGR
    "FireExt": (0, 255, 0)   # 绿色 BGR
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]

class_ids = [0, 0, 1, 1, 0, 0]
detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]
    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)
    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00  # 模拟置信度
    })


# 1. 读取图像【读取图像，首先我们编写读取图片数据的代码用于读取指定路径中事先准备好的图像】
orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    # 替换为您原先环境下的返回值
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # 2. 复制图像用于绘制【为了不损坏原始图像数据，我们利用代码复制一份相同的数据用来进行图像绘制操作】
    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 绘制检测结果
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框（注意：可能是 'bbox' 或 'box'）
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别信息
        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        # 获取类别名称
        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        # 获取颜色
        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            # 使用随机颜色或默认绿色
            color = (0, 255, 0)

        # 绘制边界框（将线条粗细从 2 改为 4）
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 10)  # ← 这里改成了 4

        # 绘制标签
        label = f"{class_name}: {confidence:.2f}"
        # 获取文本尺寸
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)

        # 绘制标签背景
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 10), (x1 + text_w, y1), color, -1)
        # 绘制标签文字
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    # 4. 转换回 RGB 用于显示【为了能让检测完毕的图像能够进行显示，将文件转回为RGB色彩空间，我们进行用于转换的代码编写】
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 显示图像【文件已转换完毕，下面进行显示图像的代码编写，这5行代码实现了 画布创建 → 图像渲染 → 界面美化 → 信息标注 → 最终呈现 的完整显示流程，是可视化模块中连接数据处理与人眼观察的关键环节】
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    # 6. 保存结果【显示图像的代码已编写完成，下面将结果保存，首先定义输出路径，其次将路径输出的图片保存到指定文件，最后返回图像路径，这4行代码实现了 保存结果到文件 和 返回数据给调用方 两个功能】
    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

# 返回变量 (如果是在函数里)
# return img_bgr

In [ ]:
import cv2
import matplotlib.pyplot as plt


image_path = r".\inference\test_image.jpg"
categories = ["GasCyl", "FireExt"]
class_colors = {
    "GasCyl": (0, 0, 255),   # 红色 BGR
    "FireExt": (0, 255, 0)   # 绿色 BGR
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]

class_ids = [0, 0, 1, 1, 0, 0]
detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]
    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)
    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00  # 模拟置信度
    })


# 1. 读取图像【读取图像，首先我们编写读取图片数据的代码用于读取指定路径中事先准备好的图像】
orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    # 替换为您原先环境下的返回值
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # 2. 复制图像用于绘制【为了不损坏原始图像数据，我们利用代码复制一份相同的数据用来进行图像绘制操作】
    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 绘制检测结果
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框（注意：可能是 'bbox' 或 'box'）
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别信息
        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        # 获取类别名称
        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        # 获取颜色
        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            # 使用随机颜色或默认绿色
            color = (0, 255, 0)

        # 绘制边界框
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 4)

        # 绘制标签（字体大小从 0.5 改为 0.8，文字粗细从 2 改为 3）
        label = f"{class_name}: {confidence:.2f}"
        # 获取文本尺寸（字体大小改为 0.8）
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 3, 10)

        # 绘制标签背景（根据新的文字尺寸调整背景框）
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 12), (x1 + text_w, y1 + 4), color, -1)
        # 绘制标签文字（字体大小改为 0.8，文字粗细改为 3）
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 3)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    # 4. 转换回 RGB 用于显示【为了能让检测完毕的图像能够进行显示，将文件转回为RGB色彩空间，我们进行用于转换的代码编写】
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 显示图像【文件已转换完毕，下面进行显示图像的代码编写，这5行代码实现了 画布创建 → 图像渲染 → 界面美化 → 信息标注 → 最终呈现 的完整显示流程，是可视化模块中连接数据处理与人眼观察的关键环节】
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    # 6. 保存结果【显示图像的代码已编写完成，下面将结果保存，首先定义输出路径，其次将路径输出的图片保存到指定文件，最后返回图像路径，这4行代码实现了 保存结果到文件 和 返回数据给调用方 两个功能】
    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

# 返回变量 (如果是在函数里)
# return img_bgr

In [ ]:
import cv2
import matplotlib.pyplot as plt


image_path = r".\inference\test_image.jpg"
categories = ["GasCyl", "FireExt"]
class_colors = {
    "GasCyl": (0, 0, 255),   # 红色 BGR
    "FireExt": (0, 255, 0)   # 绿色 BGR
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]

class_ids = [0, 0, 1, 1, 0, 0]
detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]
    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)
    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00  # 模拟置信度
    })


# 1. 读取图像【读取图像，首先我们编写读取图片数据的代码用于读取指定路径中事先准备好的图像】
orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    # 替换为您原先环境下的返回值
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # 2. 复制图像用于绘制【为了不损坏原始图像数据，我们利用代码复制一份相同的数据用来进行图像绘制操作】
    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 绘制检测结果
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框（注意：可能是 'bbox' 或 'box'）
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别信息
        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        # 获取类别名称
        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        # 获取颜色
        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            # 使用随机颜色或默认绿色
            color = (0, 255, 0)

        # 绘制边界框
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 4)

        # 绘制标签（只显示类别名称）
        label = class_name
        # 获取文本尺寸（字体大小 1.5，文字粗细 4）
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 4)

        # 绘制标签背景
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 18), (x1 + text_w, y1 + 8), color, -1)
        # 绘制标签文字（字体大小 1.5，文字粗细 4）
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    # 4. 转换回 RGB 用于显示【为了能让检测完毕的图像能够进行显示，将文件转回为RGB色彩空间，我们进行用于转换的代码编写】
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 显示图像【文件已转换完毕，下面进行显示图像的代码编写，这5行代码实现了 画布创建 → 图像渲染 → 界面美化 → 信息标注 → 最终呈现 的完整显示流程，是可视化模块中连接数据处理与人眼观察的关键环节】
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    # 6. 保存结果【显示图像的代码已编写完成，下面将结果保存，首先定义输出路径，其次将路径输出的图片保存到指定文件，最后返回图像路径，这4行代码实现了 保存结果到文件 和 返回数据给调用方 两个功能】
    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

# 返回变量 (如果是在函数里)
# return img_bgr

In [ ]:
import cv2
import matplotlib.pyplot as plt


image_path = r".\inference\test_image.jpg"
categories = ["GasCyl", "FireExt"]
class_colors = {
    "GasCyl": (0, 0, 255),   # 红色 BGR
    "FireExt": (0, 255, 0)   # 绿色 BGR
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]

class_ids = [0, 0, 1, 1, 0, 0]
detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]
    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)
    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00  # 模拟置信度
    })


# 1. 读取图像【读取图像，首先我们编写读取图片数据的代码用于读取指定路径中事先准备好的图像】
orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    # 替换为您原先环境下的返回值
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # 2. 复制图像用于绘制【为了不损坏原始图像数据，我们利用代码复制一份相同的数据用来进行图像绘制操作】
    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 绘制检测结果
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框（注意：可能是 'bbox' 或 'box'）
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别信息
        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        # 获取类别名称
        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        # 获取颜色
        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            # 使用随机颜色或默认绿色
            color = (0, 255, 0)

        # 绘制边界框（线条粗细从 4 改为 5）
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 5)

        # 绘制标签（只显示类别名称）
        label = class_name
        # 获取文本尺寸（字体大小 1.5，文字粗细 4）
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 4)

        # 绘制标签背景
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 18), (x1 + text_w, y1 + 8), color, -1)
        # 绘制标签文字（字体大小 1.5，文字粗细 4）
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    # 4. 转换回 RGB 用于显示【为了能让检测完毕的图像能够进行显示，将文件转回为RGB色彩空间，我们进行用于转换的代码编写】
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 显示图像【文件已转换完毕，下面进行显示图像的代码编写，这5行代码实现了 画布创建 → 图像渲染 → 界面美化 → 信息标注 → 最终呈现 的完整显示流程，是可视化模块中连接数据处理与人眼观察的关键环节】
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    # 6. 保存结果【显示图像的代码已编写完成，下面将结果保存，首先定义输出路径，其次将路径输出的图片保存到指定文件，最后返回图像路径，这4行代码实现了 保存结果到文件 和 返回数据给调用方 两个功能】
    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

# 返回变量 (如果是在函数里)
# return img_bgr

In [ ]:
import cv2
import matplotlib.pyplot as plt


image_path = r".\inference\test_image.jpg"
categories = ["GasCyl", "FireExt"]
class_colors = {
    "GasCyl": (0, 0, 255),   # 红色 BGR
    "FireExt": (0, 255, 0)   # 绿色 BGR
}

centers = [(798, 603), (1223, 900), (1282, 1461), (2521, 543), (3275, 1032), (3376, 1347)]
sizes = [(220, 250), (230, 230), (220, 240), (200, 220), (210, 240), (240, 260)]

# 调整第一个框往右移（x坐标 +50），第六个框往左移（x坐标 -50）
centers[0] = (848, 603)   # 第一个框：798 -> 848，往右移50像素
centers[5] = (3326, 1347) # 第六个框：3376 -> 3326，往左移50像素

class_ids = [0, 0, 1, 1, 0, 0]
detections = []
for i in range(6):
    x_c, y_c = centers[i]
    w, h = sizes[i]
    x1 = int(x_c - w / 2)
    y1 = int(y_c - h / 2)
    x2 = int(x_c + w / 2)
    y2 = int(y_c + h / 2)
    detections.append({
        'bbox': (x1, y1, x2, y2),
        'class_id': class_ids[i],
        'confidence': 1.00  # 模拟置信度
    })


# 1. 读取图像【读取图像，首先我们编写读取图片数据的代码用于读取指定路径中事先准备好的图像】
orig_img = cv2.imread(image_path)
if orig_img is None:
    print(f"❌ 无法读取图像: {image_path}")
    # 替换为您原先环境下的返回值
    img_bgr = None
else:
    print(f"✅ 读取图像: {image_path}")
    print(f"   图像尺寸: {orig_img.shape}")

    # 2. 复制图像用于绘制【为了不损坏原始图像数据，我们利用代码复制一份相同的数据用来进行图像绘制操作】
    img_bgr = orig_img.copy()
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 3. 绘制检测结果
    print(f"\n📊 绘制 {len(detections)} 个检测目标")

    for i, det in enumerate(detections):
        # 获取边界框（注意：可能是 'bbox' 或 'box'）
        if 'bbox' in det:
            x1, y1, x2, y2 = det['bbox']
        elif 'box' in det:
            x1, y1, x2, y2 = det['box']
        else:
            print(f"⚠️  检测目标 {i} 没有边界框信息")
            continue

        # 获取类别信息
        class_id = det.get('class_id', 0)
        confidence = det.get('confidence', 0.0)

        # 获取类别名称
        if class_id < len(categories):
            class_name = categories[class_id]
        else:
            class_name=f"Class_{class_id}"

        # 获取颜色
        if class_colors and class_name in class_colors:
            color = class_colors[class_name]
        else:
            # 使用随机颜色或默认绿色
            color = (0, 255, 0)

        # 绘制边界框
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color, 5)

        # 绘制标签（只显示类别名称）
        label = class_name
        # 获取文本尺寸（字体大小 1.5，文字粗细 4）
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 1.5, 4)

        # 绘制标签背景
        cv2.rectangle(img_bgr, (x1, y1 - text_h - 18), (x1 + text_w, y1 + 8), color, -1)
        # 绘制标签文字（字体大小 1.5，文字粗细 4）
        cv2.putText(img_bgr, label, (x1, y1 - 5),
                   cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 4)

        print(f"   [{i+1}] {class_name}: conf={confidence:.3f}, box=({x1},{y1},{x2},{y2})")

    # 4. 转换回 RGB 用于显示【为了能让检测完毕的图像能够进行显示，将文件转回为RGB色彩空间，我们进行用于转换的代码编写】
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    # 5. 显示图像【文件已转换完毕，下面进行显示图像的代码编写，这5行代码实现了 画布创建 → 图像渲染 → 界面美化 → 信息标注 → 最终呈现 的完整显示流程，是可视化模块中连接数据处理与人眼观察的关键环节】
    plt.figure(figsize=(12, 8))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'检测结果: {len(detections)} 个目标')
    plt.show()

    # 6. 保存结果【显示图像的代码已编写完成，下面将结果保存，首先定义输出路径，其次将路径输出的图片保存到指定文件，最后返回图像路径，这4行代码实现了 保存结果到文件 和 返回数据给调用方 两个功能】
    output_path = 'detection_result.jpg'
    cv2.imwrite(output_path, img_bgr)
    print(f"\n✅ 结果已保存到: {output_path}")

# 返回变量 (如果是在函数里)
# return img_bgr